## Notebook Setup: Hugging Face Token and Model Repositories

This section initializes the Hugging Face token and defines the names of the base model and various fine-tuned model repositories. Ensure your `HF_TOKEN` is set for pushing models to the Hugging Face Hub.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

MODEL_TO_FINE_TUNE = "unsloth/medgemma-4b-it"

REPO_FIRST_TUNE = "AIOL-HQ/MedGemma-4b-it-finetuned-V1"
REPO_2ND_TUNE = "AIOL-HQ/MedGemma-4b-it-finetuned-V2"
REPO_GRPO_TUNE = "AIOL-HQ/MedGemma-4b-it-finetuned-GRPO"
REPO_DPO_TUNE = "AIOL-HQ/MedGemma-4b-it-finetuned-DRO"

## Environment Setup: Install Necessary Packages

This cell installs all the required Python libraries, including `unsloth`, `transformers`, `bitsandbytes`, `accelerate`, `peft`, and `trl`. It checks if the notebook is running in Colab to optimize installation.

In [ ]:
%%capture
import os, re
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.3
!pip install --no-deps trl==0.22.2

## Model Initialization: Load Base MedGemma Model

This section loads the `unsloth/medgemma-4b-it` model using `FastVisionModel.from_pretrained`. It configures the model to load in 4-bit quantization to reduce memory usage and enables gradient checkpointing for efficient training.

In [ ]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

model, processor = FastVisionModel.from_pretrained(
    MODEL_TO_FINE_TUNE,
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

## PEFT Configuration: Prepare Model for Fine-tuning

Here, the model is prepared for fine-tuning using Parameter-Efficient Fine-tuning (PEFT) techniques, specifically LoRA (Low-Rank Adaptation). It enables fine-tuning of vision, language, attention, and MLP layers to optimize performance while minimizing trainable parameters.

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,                           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,                  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,               # We support rank stabilized LoRA
    loftq_config = None,              # And LoftQ
    target_modules = "all-linear",    # Optional now! Can specify a list if needed
)

## Tokenizer Setup: Initialize Chat Template

This cell initializes the tokenizer for the model, applying the 'gemma-3' chat template. This ensures that input prompts are correctly formatted for the model's expected conversational structure during training and inference.

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    processor,
    chat_template = "gemma-3",
)

## Data Preparation: Load and Format Medical Datasets

This section loads multiple medical datasets from Hugging Face, including text-based and image-based datasets. It then defines and applies functions to clean and standardize these datasets into a unified format (`input`, `output`, `image`) for multimodal training.

In [ ]:
from datasets import load_dataset, concatenate_datasets, Image

# --- 1. Load each dataset ---
dataset_1 = load_dataset("Ahmed-Selem/Shifaa_Arabic_Medical_Consultations", split="train")
dataset_2 = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", 'en', split="train")
dataset_3 = load_dataset("RecurvAI/Recurv-Medical-Dataset", split="train")
# ROCOv2 contains 'image' and 'caption'
dataset_4 = load_dataset("unsloth/Radiology_mini", split="train")
dataset_5 = load_dataset("Lunzima/Radiology", split="train")
# --- 2. Define Extraction Functions ---

# For Text Datasets: Add a None image column
def format_text_ds1(example):
    return {"input": example["Question"], "output": example["Answer"], "image": None}

def format_text_ds2(example):
    return {"input": example["Question"], "output": example["Response"], "image": None}

def format_text_ds3(example):
    return {"input": example["input"], "output": example["output"], "image": None}

# For Image Datasets with 'image' and 'caption' columns (like dataset_4)
def format_image_ds_caption(example):
    # For datasets with 'caption' column
    try:
        return {
            "input": "Describe this medical image.",
            "output": example["caption"],
            "image": example["image"] # This keeps the PIL Image object
        }
    except KeyError as e:
        print(f"KeyError in format_image_ds_caption: {e}. Available keys: {example.keys()}")
        raise

# For Image Datasets with 'image', 'question', and 'answer' columns (like dataset_5)
def format_image_ds_qa(example):
    # For datasets with 'question' and 'answer' columns
    try:
        return {
            "input": example["question"],
            "output": example["answer"],
            "image": example["image"] # This keeps the PIL Image object
        }
    except KeyError as e:
        print(f"KeyError in format_image_ds_qa: {e}. Available keys: {example.keys()}")
        raise

# --- 3. Apply Mapping ---
# Important: We cast the image column to the Image feature type so they match
dataset_1_clean = dataset_1.map(format_text_ds1, remove_columns=dataset_1.column_names, load_from_cache_file=False)
dataset_2_clean = dataset_2.map(format_text_ds2, remove_columns=dataset_2.column_names, load_from_cache_file=False)
dataset_3_clean = dataset_3.map(format_text_ds3, remove_columns=dataset_3.column_names, load_from_cache_file=False)
dataset_4_clean = dataset_4.map(format_image_ds_caption, remove_columns=dataset_4.column_names, load_from_cache_file=False)
dataset_5_clean = dataset_5.map(format_image_ds_qa, remove_columns=dataset_5.column_names, load_from_cache_file=False)
# --- 4. Merge and Shuffle ---
# We ensure all datasets follow the same feature schema
merged_dataset = concatenate_datasets([
    dataset_1_clean,
    dataset_2_clean,
    dataset_3_clean,
    dataset_4_clean,
    dataset_5_clean
])

med_dataset = merged_dataset.shuffle(seed=42)

# --- 5. Verify ---
print(f"Total samples: {len(med_dataset)}")
print(f"First sample keys: {med_dataset[0].keys()}")
# Check if the first image exists (it might be None if it's a text sample)
print(f"Sample 0 Image: {med_dataset[0]['image']}")

## Standardize Data Formats

This cell uses `unsloth.chat_templates.standardize_data_formats` to further process the merged dataset, ensuring consistency in column names and data types across all samples, which is crucial for subsequent training steps.

In [ ]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(med_dataset)

## Prompt Formatting: Prepare Data for Multimodal Training

This cell defines a `formatting_prompts_func` that structures the dataset examples into the model's expected multimodal chat format. It differentiates between text-only and image-with-text inputs, applying the tokenizer's chat template. The function returns both text and image columns, which is critical for multimodal models.

In [ ]:
def formatting_prompts_func(examples):
    # 1. Get the lists from the batch
    # Unsloth's standardize_data_formats usually renames 'input' to 'instruction'
    # We use .get() to be safe if the column is named 'input' or 'instruction'
    inputs  = examples.get('instruction', examples.get('input'))
    outputs = examples['output']

    # CRITICAL: Retrieve the images. If a row has no image, we use None.
    images  = examples.get('image', [None] * len(inputs))

    texts = []
    for i in range(len(inputs)):
        # 2. Construct the User Content (Multimodal Format)
        if images[i] is not None:
            # This tells the model: "Here is an image, and here is the text."
            user_content = [{"type": "image"}, {"type": "text", "text": inputs[i]}]
        else:
            # Pure text example (no image)
            user_content = inputs[i]

        conversation = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": outputs[i]} # Use 'assistant', not 'model'
        ]

        # 3. Apply the Chat Template
        formatted_text = tokenizer.apply_chat_template(
            conversation,
            tokenize = False,
            add_generation_prompt = False
        )
        # Remove the BOS token if it's already added by the trainer
        texts.append(formatted_text.lstrip('<|begin_of_text|>').lstrip('<bos>'))

    # 4. CRITICAL: Return BOTH text and image columns
    return { "text" : texts, "image": images }

# Apply the mapping
dataset = dataset.map(formatting_prompts_func, batched = True)

## Trainer Initialization: Configure SFTTrainer

This section initializes the `SFTTrainer` (Supervised Fine-tuning Trainer) from the `trl` library. It sets up training parameters such as batch size, gradient accumulation steps, learning rate, and optimizer. `max_steps` is set to -1 for a full training run.

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 1,
        max_steps = -1, # SET TO -1 FOR A FULL TRAIN
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs # DEFULT 2e-4
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

## Training Optimization: Train on Responses Only

This cell applies a utility function from `unsloth.chat_templates` to modify the trainer. This optimization ensures that the model's loss is calculated and backpropagated only on the generated assistant responses, rather than on the entire prompt, making training more efficient and focused.

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

## Model Training: Start Fine-tuning Process

This cell initiates the fine-tuning process by calling `trainer.train()`. The model will now train on the prepared dataset according to the configurations set in the `SFTTrainer`.

In [ ]:
trainer_stats = trainer.train()

## Inference Demonstration 1: Text-only Query

This section demonstrates how to perform inference with the fine-tuned model using a text-only input. It applies the chat template to format the user's question, tokenizes it, and then uses the model to generate a response. The output is then decoded and printed.

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "What's the active indgredent in paracetamol?",
    }]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)
outputs = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

## Inference Demonstration 2: Text-only Query with Streaming

Similar to the previous cell, this demonstrates text-only inference but utilizes a `TextStreamer` for a more interactive, streaming output. This is useful for observing the model's generation process in real-time.

In [ ]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "Why is the sky blue?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)

from transformers import TextStreamer
_ = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

## Save Fine-tuned Model

This cell allows you to save the fine-tuned model as a merged model. If `True`, it saves the model and tokenizer to a local directory, making it ready for deployment or further use.

In [ ]:
if True: # Change to True to save finetune!
    model.save_pretrained_merged("MedGemma-4B-it-finetuned", tokenizer)

## Push Model to Hugging Face Hub

This cell provides functionality to push the fine-tuned model to the Hugging Face Hub. If `True`, the merged model and tokenizer are uploaded to the specified repository, making it publicly or privately accessible.

In [ ]:
if True: # Change to True to upload finetune
    model.push_to_hub_merged(
        REPO_FIRST_TUNE, tokenizer,
        token = HF_TOKEN
    )

## Phase 2 Setup: Load Fine-tuned Model for Further Training

This section frees up GPU memory from the first phase of training and loads the previously fine-tuned model (`REPO_FIRST_TUNE`) as the base for the second phase. The model is loaded with 4-bit quantization and gradient checkpointing enabled to optimize resource usage.

In [ ]:
# Free Phase 1 model from GPU memory
import gc, torch

del model, processor, tokenizer, trainer_p1, dataset
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared. Loading Phase 2 base model...")

from unsloth import FastVisionModel

model, processor = FastVisionModel.from_pretrained(
    REPO_FIRST_TUNE,
    load_in_4bit               = True,
    use_gradient_checkpointing = "unsloth",
)
print("Phase 2 model loaded.")

## Phase 2 PEFT Configuration: Prepare Model for Text-only Fine-tuning

This cell configures the model for Parameter-Efficient Fine-tuning (PEFT) in Phase 2. Since the subsequent training data (MedQA) is text-only, vision layers are excluded from fine-tuning, focusing instead on language, attention, and MLP layers.

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # MedQA is text-only
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r              = 16,
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    random_state   = 3407,
    use_rslora     = False,
    loftq_config   = None,
    target_modules = "all-linear",
)

## Phase 2 Tokenizer Setup: Initialize Chat Template

This cell initializes the tokenizer for the Phase 2 model, applying the 'gemma-3' chat template. This ensures that the text-based input prompts for the MedQA dataset are correctly formatted for the model's conversational structure.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    processor,
    chat_template = "gemma-3",
)

## Phase 2 Data Preparation: Load and Format MedQA Dataset

This section loads the MedQA dataset, which is a multiple-choice question-answering dataset. It defines a `format_as_conversation` function to transform the questions and options into a conversational format suitable for the model's chat template.

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("GBaker/MedQA-USMLE-4-options", split="train")

def format_as_conversation(example):
    options_str = "\n".join([f"{k}: {v}" for k, v in example['options'].items()])
    user_message = f"{example['question']}\n\nOptions:\n{options_str}"

    correct_label = next(
        (k for k, v in example['options'].items() if v == example['answer']), None
    )
    assistant_message = f"{correct_label}: {example['answer']}" if correct_label else example['answer']

    conversation = [
        {"role": "user",      "content": user_message},
        {"role": "assistant", "content": assistant_message},
    ]
    text = tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=False,
    )
    text = text.lstrip('<bos>').lstrip('<|begin_of_text|>')
    return {"text": text}

dataset = raw_dataset.map(
    format_as_conversation,
    remove_columns      = raw_dataset.column_names,
    load_from_cache_file = False,
)
print(f"Total samples: {len(dataset)}")
print("\n--- Sample conversation ---")
print(dataset[0]['text'])

## Phase 2 Trainer Initialization: Configure SFTTrainer

This section initializes the `SFTTrainer` for the second phase of fine-tuning. It sets specific training arguments, including batch size, gradient accumulation steps, learning rate, and the number of training steps, optimized for the MedQA dataset.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer_p2 = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    eval_dataset  = None,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # effective batch = 8
        warmup_steps                = 10,
        num_train_epochs            = 1,
        max_steps                   = -1,  # -1 for full epoch
        learning_rate               = 2e-4,
        logging_steps               = 10,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = 3407,
        output_dir                  = "hakeem_medqa_checkpoints",
        report_to                   = "none",
    ),
)

## Phase 2 Training Optimization: Train on Responses Only

This cell applies an optimization to the `SFTTrainer` for Phase 2, ensuring that the model's loss calculation and backpropagation are focused exclusively on the generated assistant responses, improving training efficiency and target alignment.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer_p2 = train_on_responses_only(
    trainer_p2,
    instruction_part = "<start_of_turn>user\n",
    response_part    = "<start_of_turn>model\n",
)

## Phase 2 Model Training: Start Fine-tuning Process

This cell initiates the second phase of the fine-tuning process by calling `trainer_p2.train()`. The model will now continue training on the prepared MedQA dataset according to the configurations set in the `SFTTrainer` for Phase 2.

In [ ]:
trainer_stats_p2 = trainer_p2.train()
print(trainer_stats_p2)

## Phase 2 Inference Demonstration: Text-only Medical Query

This section demonstrates how to perform inference with the model after Phase 2 fine-tuning. It uses a medical question from the MedQA domain and streams the model's generated response in real-time.

In [ ]:
from transformers import TextStreamer

FastVisionModel.for_inference(model)

test_question = (
    "A 45-year-old woman presents with fatigue, weight gain, and cold intolerance. "
    "TSH is elevated and free T4 is low. What is the most appropriate treatment?\n\n"
    "Options:\nA: Methimazole\nB: Levothyroxine\nC: Propylthiouracil\nD: Radioactive iodine"
)
messages = [{"role": "user", "content": test_question}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize              = True,
    return_tensors        = "pt",
    return_dict           = True,
).to("cuda")

_ = model.generate(
    **inputs,
    max_new_tokens = 64,
    temperature    = 1.0, top_p = 0.95, top_k = 64,
    streamer       = TextStreamer(tokenizer, skip_prompt=True),
)

## Push Phase 2 Model to Hugging Face Hub

This cell provides functionality to push the fine-tuned model from Phase 2 to the Hugging Face Hub. The merged model and tokenizer are uploaded to the specified repository (`REPO_2ND_TUNE`), making the second-stage fine-tuned model publicly or privately accessible.

In [ ]:
model.push_to_hub_merged(
    REPO_2ND_TUNE,
    tokenizer,
    token = HF_TOKEN,
)
print("Phase 2 upload complete → REPO_2ND_TUNE")

In [ ]:
import gc, torch
import re
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel, PatchFastRL, FastModel
from unsloth import is_bfloat16_supported
from trl import GRPOConfig, GRPOTrainer

# Unsloth's PatchFastRL is required to make GRPO memory efficient and fast
PatchFastRL("GRPO", FastLanguageModel)
del model, processor, tokenizer, trainer_p1, dataset
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared. Loading Phase 3 base model...")




max_seq_length = 1024
model, tokenizer = FastModel.from_pretrained(
    model_name = REPO_2ND_TUNE,
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

## GRPO Setup: Prompts and Reward Logic

In this section, we define the special tokens for reasoning and the system prompt. We also implement the reward functions that will score the model based on whether it follows the required format and provides the correct medical answer.

In [ ]:
import re
from datasets import load_dataset

# 1. Define formatting tokens
reasoning_start = "<start_working_out>"
reasoning_end   = "<end_working_out>"
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

system_prompt = f"""You are given a medical problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your final answer (just the letter A, B, C, or D) between {solution_start} and {solution_end}."""

# 2. Reward Function: Format Check
def match_format_reward(completions, **kwargs):
    pattern = rf"{reasoning_start}.+?{reasoning_end}.*?{solution_start}.+?{solution_end}"
    scores = []
    for completion in completions:
        content = completion[0]["content"]
        score = 2.0 if re.search(pattern, content, re.DOTALL) else 0.0
        scores.append(score)
    return scores

# 3. Reward Function: Accuracy Check
def check_answer_reward(prompts, completions, answer, **kwargs):
    scores = []
    for completion, true_answer in zip(completions, answer):
        content = completion[0]["content"]
        # Extract the content inside <SOLUTION> tags
        match = re.search(rf"{solution_start}\s*([A-D])\s*{solution_end}", content)
        if match:
            predicted = match.group(1).strip()
            scores.append(3.0 if predicted == true_answer.strip() else 0.0)
        else:
            scores.append(0.0)
    return scores

## Dataset Preparation for GRPO

We load the MedQA dataset and map it to a format compatible with GRPO, including the system prompt and the extraction of the correct option label (A, B, C, or D) for the reward function.

In [ ]:
raw_dataset = load_dataset("GBaker/MedQA-USMLE-4-options", split="train")

def prepare_grpo_data(example):
    options_str = "\n".join([f"{k}: {v}" for k, v in example['options'].items()])
    question_text = f"{example['question']}\n\nOptions:\n{options_str}"

    # Find the letter (key) corresponding to the correct answer string
    correct_label = next((k for k, v in example['options'].items() if v == example['answer']), "")

    return {
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question_text}
        ],
        "answer": correct_label
    }

dataset = raw_dataset.map(prepare_grpo_data)

## GRPO Trainer Configuration

We initialize the `GRPOTrainer` with specific configurations for the group size, number of generations, and the reward functions defined earlier. This setup allows the model to learn by comparing multiple generated responses for the same prompt.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

max_prompt_length = 256
# Note: max_seq_length was defined as 1024 earlier

training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_torch_fused",
    logging_steps = 1,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1,
    num_generations = 4,
    max_prompt_length = max_prompt_length,
    max_completion_length = max_seq_length - max_prompt_length,
    max_steps = 50,
    save_steps = 50,
    max_grad_norm = 0.1,
    report_to = "none",
    output_dir = "medgemma-grpo-checkpoints",
)

trainer_grpo = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_reward,
        check_answer_reward,
        # Assuming matching logic from your snippet for extra robustness
    ],
    args = training_args,
    train_dataset = dataset,
)

trainer_grpo.train()

## Save and Push GRPO Model

This final section handles saving the model after the GRPO training phase and pushing the resulting weights to the Hugging Face Hub using the `REPO_GRPO_TUNE` identifier.

In [ ]:
if True: # Set to True to save the GRPO fine-tune locally
    model.save_pretrained_merged("MedGemma-GRPO-finetuned", tokenizer)

if True: # Set to True to upload the GRPO fine-tune to HF Hub
    model.push_to_hub_merged(
        REPO_GRPO_TUNE,
        tokenizer,
        token = HF_TOKEN
    )
print(f"GRPO Training Phase Complete. Model pushed to {REPO_GRPO_TUNE}")

In [ ]:
import gc, torch

del model, processor, tokenizer, trainer_p1, dataset
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared. Loading Phase 4 base model...")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from unsloth import FastLanguageModel, PatchDPOTrainer
from unsloth import is_bfloat16_supported

# MUST be called before importing DPOTrainer
PatchDPOTrainer()

import torch
from trl import DPOTrainer, DPOConfig

max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name  = REPO_GRPO_TUNE,
    max_seq_length = max_seq_length,
    dtype       = None,         # Auto-detect: bf16 if supported, else fp16
    load_in_4bit = True,
)
print("Model loaded.")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 64,   # Higher r = more capacity for preference learning
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 64,
    lora_dropout   = 0,    # 0 is optimized per docs
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
    max_seq_length = max_seq_length,
)

## Step 4 — Build DPO Dataset from MedQA

DPO needs three columns: `prompt`, `chosen`, `rejected`.

We build these directly from MedQA — no separate dataset needed:
- **prompt** — the clinical question with options, as a chat message list
- **chosen** — the correct answer (what the model *should* say)
- **rejected** — a wrong answer (what the model *should not* say)

> Both chosen and rejected are formatted as assistant messages so DPO can compare them.

In [ ]:
import random
from datasets import load_dataset

random.seed(42)

raw = load_dataset("GBaker/MedQA-USMLE-4-options", split="train")

SYSTEM_PROMPT = (
    "You are Hakeem, an expert medical AI assistant. "
    "Answer the clinical question by selecting the single best option."
)

def build_dpo_sample(example):
    options_str = "\n".join([f"{k}: {v}" for k, v in example['options'].items()])
    question_text = f"{example['question']}\n\nOptions:\n{options_str}"

    # Find the correct label and its text
    correct_label = next(
        (k for k, v in example['options'].items() if v == example['answer']), None
    )

    # Pick one wrong option as the rejected answer
    wrong_options = [
        (k, v) for k, v in example['options'].items() if v != example['answer']
    ]
    wrong_label, wrong_text = random.choice(wrong_options)

    return {
        # prompt: list of chat messages (system + user)
        "prompt": [
            {"role": "system",  "content": SYSTEM_PROMPT},
            {"role": "user",    "content": question_text},
        ],
        # chosen: correct answer as assistant message
        "chosen": [
            {"role": "assistant", "content": f"{correct_label}: {example['answer']}"}
        ],
        # rejected: wrong answer as assistant message
        "rejected": [
            {"role": "assistant", "content": f"{wrong_label}: {wrong_text}"}
        ],
    }

dataset = raw.map(
    build_dpo_sample,
    remove_columns = raw.column_names,
    load_from_cache_file = False,
)

print(f"DPO dataset size: {len(dataset)}")
print("\n--- Sample ---")
print("PROMPT:",   dataset[0]['prompt'])
print("CHOSEN:",   dataset[0]['chosen'])
print("REJECTED:", dataset[0]['rejected'])

## Step 5 — Configure DPO Trainer

Key settings straight from the Unsloth docs:

| Parameter | Value | Why |
|---|---|---|
| `ref_model = None` | — | Unsloth handles the reference model internally |
| `beta = 0.1` | 0.1 | KL penalty — how much to stay close to the original model |
| `max_prompt_length` | 512 | Leaves room for the answer in the 1024 context |
| `max_length` | 1024 | Full context window |

In [ ]:
dpo_trainer = DPOTrainer(
    model     = model,
    ref_model = None,   # Unsloth handles reference model — do not set
    args = DPOConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 8,   # effective batch = 32
        warmup_ratio                = 0.1,
        num_train_epochs            = 3,
        fp16  = not is_bfloat16_supported(),
        bf16  = is_bfloat16_supported(),
        logging_steps               = 1,
        optim = "adamw_8bit",
        seed  = 42,
        output_dir = "hakeem_dpo_checkpoints",
        report_to  = "none",
    ),
    beta              = 0.1,           # KL penalty coefficient
    train_dataset     = dataset,
    tokenizer         = tokenizer,
    max_length        = 1024,
    max_prompt_length = 512,
)
print("DPO trainer configured.")

## Step 6 — Train

> DPO is much faster than GRPO — no rollout generation needed.  
> 3 epochs on MedQA should take a couple of hours on a T4.

In [ ]:
dpo_trainer.train()

## Step 7 — Quick Sanity Check

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

test_question = (
    "A 60-year-old man presents with crushing chest pain radiating to the left arm, "
    "diaphoresis, and shortness of breath. ECG shows ST elevation in leads II, III, aVF. "
    "What is the most appropriate immediate intervention?\n\n"
    "Options:\n"
    "A: Oral aspirin and discharge\n"
    "B: Primary PCI (percutaneous coronary intervention)\n"
    "C: IV furosemide\n"
    "D: Thrombolysis with streptokinase only if PCI unavailable within 120 minutes"
)

messages = [
    {"role": "system",  "content": SYSTEM_PROMPT},
    {"role": "user",    "content": test_question},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize     = True,
    return_tensors = "pt",
    return_dict  = True,
).to("cuda")

_ = model.generate(
    **inputs,
    max_new_tokens = 128,
    temperature    = 1.0, top_p = 0.95, top_k = 64,
    streamer       = TextStreamer(tokenizer, skip_prompt=True),
)

## Step 8 — Push to Hub


In [ ]:

model.push_to_hub_merged(
    REPO_DPO_TUNE,
    tokenizer,
    save_method = "merged_16bit",
    token       = HF_TOKEN,
)
print("Done → ", {REPO_DPO_TUNE})